# Atelier Prompt Engineering — AI Business Assistant

**Contexte** : une entreprise souhaite mettre en place un « AI Business Assistant », un assistant IA polyvalent capable d'aider ses collaborateurs à exploiter des documents, analyser des données, faire du machine learning et produire des résultats structurés.

## Partie 1 — Anatomie d'un prompt

### Prompt

> Tu es un analyste satisfaction client senior pour une entreprise de services. Contexte : je te fournis un ensemble d'avis clients bruts collectés sur plusieurs canaux (email, réseaux sociaux, support). Tâche : analyse ces avis pour identifier le sentiment général, les thèmes récurrents (positifs et négatifs) et les points d'amélioration prioritaires. Contraintes : appuie chaque conclusion sur des éléments présents dans les avis, ne fais aucune supposition non justifiée, reste factuel et synthétique. Format de sortie : un résumé en 3 parties (Sentiment global, Thèmes récurrents, Recommandations), chacune sous forme de liste à puces.


### Composantes identifiées

| Composante | Contenu |
|---|---|
| **Rôle** | Analyste satisfaction client senior |
| **Contexte** | Avis clients bruts multi-canaux (email, réseaux sociaux, support) |
| **Tâche** | Identifier sentiment général, thèmes récurrents, points d'amélioration |
| **Contraintes** | S'appuyer uniquement sur les avis fournis, pas de suppositions, rester factuel |
| **Format de sortie** | 3 sections à puces : Sentiment global / Thèmes récurrents / Recommandations |


## Partie 2 — Comparer les techniques de prompting

Comparaison des techniques **zero-shot**, **one-shot**, **few-shot** et **prompt structuré** sur la classification du commentaire : *"Le service est rapide mais l'application plante régulièrement."*


### Prompt 1 — Zero-shot

> Classe le commentaire suivant dans une des catégories : positif, négatif, neutre. Réponds uniquement avec le mot de la catégorie, sans explication.
> Commentaire : "Le service est rapide mais l'application plante régulièrement."

**Réponse obtenue :** négatif

### Prompt 2 — One-shot

> Exemple : "Le personnel est agréable." -> positif
>
> Classe le commentaire suivant dans une des catégories : positif, négatif, neutre. Réponds uniquement avec le mot de la catégorie, sans explication.
> Commentaire : "Le service est rapide mais l'application plante régulièrement."

**Réponse obtenue :** négatif

### Prompt 3 — Few-shot

> Exemples :
> "Le personnel est agréable." -> positif
> "La livraison a eu 5 jours de retard." -> négatif
> "Le produit correspond à la description." -> neutre
>
> Classe le commentaire suivant dans une des catégories : positif, négatif, neutre. Réponds uniquement avec le mot de la catégorie, sans explication.
> Commentaire : "Le service est rapide mais l'application plante régulièrement."

**Réponse obtenue :** négatif

### Prompt 4 — Structuré

> Rôle : classificateur de sentiment.
> Tâche : attribuer une seule classe parmi [positif, négatif, neutre] au commentaire suivant.
> Commentaire : "Le service est rapide mais l'application plante régulièrement."
> Contrainte : si le commentaire contient à la fois un aspect positif et un aspect négatif, privilégier la classe correspondant au problème le plus bloquant pour l'usage du service.
> Format de sortie : renvoyer uniquement le mot de la classe, sans justification.

**Réponse obtenue :** négatif

### Comparaison des résultats

| Technique | Réponse | Remarque |
|---|---|---|
| Zero-shot | négatif | Aucun exemple fourni ; le modèle s'appuie uniquement sur sa compréhension générale du sentiment |
| One-shot | négatif | Un seul exemple ; aide peu à trancher un cas mixte comme celui-ci |
| Few-shot | négatif | Exemples contrastés (positif/négatif/neutre) ; renforce la cohérence du critère de classification |
| Structuré | négatif | La contrainte explicite (« privilégier le problème le plus bloquant ») lève directement l'ambiguïté du commentaire mixte |

**Analyse** : une fois le format de sortie harmonisé entre les 4 prompts, les 4 techniques convergent vers la même classe (*négatif*) avec le même format concis (un seul mot). La différence entre les techniques ne se voit donc plus sur ce commentaire (non ambigu pour le modèle), mais elle deviendrait déterminante sur un cas plus ambigu : le zero-shot resterait le moins fiable (aucun repère), le few-shot améliorerait la cohérence du critère grâce aux exemples contrastés, et le prompt structuré resterait le plus robuste car il explicite directement la règle de décision plutôt que de la laisser déduire par le modèle à partir d'exemples.

---

## Partie 3 — Prompt Engineering et raisonnement

### 3.1 — Décomposition du prompt

**Prompt soumis au LLM :**

> Décompose le prompt suivant en sous-tâches, en expliquant à quoi sert chaque étape : "Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations."

**Réponse obtenue :**

On peut décomposer ce prompt en 4 sous-tâches principales :

| # | Sous-tâche | Description | But |
|---|---|---|---|
| 1 | Analyser les avis clients | Lire chaque avis et identifier ce que le client exprime : satisfaction, insatisfaction, plainte, suggestion, etc. | Comprendre le contenu brut des avis |
| 2 | Identifier les problèmes | Regrouper les avis qui parlent du même problème (ex. retards, bugs, mauvais accueil, prix élevés) | Faire ressortir les problèmes récurrents plutôt que traiter chaque avis séparément |
| 3 | Prioriser les problèmes | Déterminer quels problèmes sont les plus importants selon leur fréquence, leur gravité ou leur impact sur le client | Savoir sur quels problèmes agir en premier |
| 4 | Formuler des recommandations | Pour chaque problème prioritaire, proposer une action permettant de le résoudre ou de le réduire | Transformer l'analyse en plan d'action concret |

**Résumé du flux :** Avis bruts → Analyse → Problèmes → Priorisation → Recommandations

**Conclusion du LLM :** le prompt initial est assez vague — il demande directement le résultat final, alors qu'un prompt plus précis pourrait demander explicitement d'effectuer chacune de ces étapes.

---


### 3.2 — Analyse avec contraintes puis auto-vérification

**Avis client analysé :**

> "La livraison a pris 6 jours au lieu de 2 annoncés. Le support a mis 3 jours à répondre à mon email. Le produit en lui-même est correct."

#### Prompt A — Analyse avec contraintes

**Prompt soumis au LLM :**

> Voici un avis client : "La livraison a pris 6 jours au lieu de 2 annoncés. Le support a mis 3 jours à répondre à mon email. Le produit en lui-même est correct." Analyse cet avis et identifie le problème principal, puis propose une recommandation. Contrainte : ne mentionne que des faits présents dans l'avis.

**Réponse obtenue :**

> **Problème principal** : la livraison a pris 6 jours au lieu des 2 jours annoncés.
>
> **Recommandation** : réduire le délai de livraison afin qu'il corresponde au délai de 2 jours annoncé.

#### Prompt B — Auto-vérification (même conversation)

**Prompt soumis au LLM :**

> Vérifie la réponse que tu viens de donner à propos de cet avis client. Contrôle : les informations non justifiées par l'avis, les contradictions, les informations absentes de l'avis mais mentionnées dans ta réponse, les éventuelles hallucinations, et le respect de la contrainte ("ne mentionner que des faits présents dans l'avis").

**Réponse obtenue :**

| Point de contrôle | Résultat |
|---|---|
| Informations justifiées | « 6 jours » et « au lieu des 2 jours annoncés » sont bien présents dans l'avis |
| Contradictions | Aucune détectée |
| Hallucinations factuelles | Aucune — aucun nouveau chiffre, événement ou problème ajouté |
| Point problématique | « afin qu'il corresponde au délai de 2 jours annoncé » transforme le délai annoncé en objectif de recommandation — ce n'est pas un fait nouveau, mais une interprétation prescriptive |

**Conclusion du LLM :** la réponse est cohérente, mais pour respecter la contrainte de manière stricte, il vaut mieux formuler la recommandation uniquement à partir des faits explicitement constatés dans l'avis, sans y injecter une reformulation prescriptive du délai annoncé.

**Analyse (atelier)** : cet exercice illustre l'intérêt d'un second prompt de vérification : même une réponse globalement correcte peut contenir une légère dérive (ici, transformer un fait en objectif normatif) que l'auto-vérification permet de détecter et de documenter, sans nécessiter d'intervention humaine.

---
